In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import time
import json
import os

### загрузка конфига

In [ ]:
def load_config(config_path='config.json'):
    with open(config_path, 'r', encoding='utf-8') as f:
        return json.load(f)

config = load_config()
params = config['parameters']
lms_cfg = params['landmarks']    

### инициализация medapipe

In [ ]:
mp_face_mesh = mp.solutions.face_mesh

### функции для анализа

In [ ]:
def get_head_pose_v2(face_landmarks, img_w, img_h):
    """Вычисляет Pitch и Yaw головы через PnP."""
    face_2d = []
    model_3d = np.array([
        [0.0, 0.0, 0.0],          # Нос
        [0.0, -330.0, -65.0],    # Подбородок
        [-225.0, 170.0, -135.0], # Левый глаз
        [225.0, 170.0, -135.0],  # Правый глаз
        [-150.0, -150.0, -125.0],# Левый рот
        [150.0, -150.0, -125.0]  # Правый рот
    ], dtype=np.float64)

    for idx in lms_cfg['pnp_indices']:
        lm = face_landmarks.landmark[idx]
        face_2d.append([lm.x * img_w, lm.y * img_h])

    face_2d = np.array(face_2d, dtype=np.float64)
    focal_length = img_w
    cam_matrix = np.array([[focal_length, 0, img_w / 2],
                           [0, focal_length, img_h / 2],
                           [0, 0, 1]])

    success, rot_vec, trans_vec = cv2.solvePnP(model_3d, face_2d, cam_matrix, np.zeros((4, 1)))
    rmat, _ = cv2.Rodrigues(rot_vec)

    pitch = np.arcsin(-rmat[2, 0])
    yaw = np.arctan2(rmat[2, 1], rmat[2, 2])
    return np.degrees(pitch), np.degrees(yaw)

In [ ]:
def get_gaze_ratio(face_landmarks, eye_indices, iris_indices):
    """Вычисляет относительное положение зрачка (0.0 - 1.0)."""
    lm = face_landmarks.landmark
    x_left = lm[eye_indices[0]].x
    x_right = lm[eye_indices[8]].x
    x_iris = np.mean([lm[i].x for i in iris_indices])
    
    width = abs(x_right - x_left)
    if width == 0: 
        return 0.5
    return (x_iris - min(x_left, x_right)) / width

In [ ]:
def analyze_gaze_advanced(face_lms, pitch, raw_yaw, params):
    """Основная логика Advanced алгоритма с подтягиванием параметров из JSON."""
    # Нормализация
    yaw = raw_yaw - params['yaw_center_offset'] if raw_yaw > 0 else raw_yaw + params['yaw_center_offset']
    
    # Расчет зрачков
    ratio_l = get_gaze_ratio(face_lms, lms_cfg['left_eye_indices'], lms_cfg['left_iris_indices'])
    ratio_r = get_gaze_ratio(face_lms, lms_cfg['right_eye_indices'], lms_cfg['right_iris_indices'])
    avg_gaze = (ratio_l + ratio_r) / 2

    # Пороги из конфига
    yaw_lim = params['head_pose_thresholds']['yaw_limit']
    p_min, p_max = params['head_pose_thresholds']['pitch_limit']
    g_min, g_max = params['iris_tracking']['center_range']
    
    is_head_centered = (abs(yaw) < yaw_lim) and (p_min < pitch < p_max)
    is_gaze_centered = (g_min < avg_gaze < g_max)
    
    # Компенсация
    compensation = False
    trigger = params['iris_tracking']['compensation_trigger_yaw']
    if yaw < -trigger and avg_gaze < params['iris_tracking']['compensation_ratio_right']:
        compensation = True
    elif yaw > trigger and avg_gaze > params['iris_tracking']['compensation_ratio_left']:
        compensation = True

    return (is_head_centered and is_gaze_centered) or compensation

### главный цикл

In [ ]:
def process_video_prototype(video_path, config):
    if not os.path.exists(video_path):
        print(f"Файл {video_path} не найден!")
        return None

    params = config['parameters']
    sampling_rate = config['processing']['sampling_rate']
    
    cap = cv2.VideoCapture(video_path)
    stats = {"hits": 0, "face_frames": 0, "total_frames": 0}
    
    start_time = time.time()

    with mp_face_mesh.FaceMesh(refine_landmarks=True, min_detection_confidence=0.5) as face_mesh:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            stats["total_frames"] += 1
            if stats["total_frames"] % sampling_rate != 0:
                continue

            h, w, _ = frame.shape
            # Опционально: ресайз для скорости как в конфиге
            # target_w = config['processing']['target_width']
            
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb_frame)

            if results.multi_face_landmarks:
                stats["face_frames"] += 1
                face_lms = results.multi_face_landmarks[0]
                
                pitch, raw_yaw = get_head_pose_v2(face_lms, w, h)
                
                if analyze_gaze_advanced(face_lms, pitch, raw_yaw, params):
                    stats["hits"] += 1

    cap.release()
    
    duration = time.time() - start_time
    score = stats["hits"] / max(1, stats["face_frames"])
    
    print(f"\n--- Результаты для {os.path.basename(video_path)} ---")
    print(f"Gaze Score: {score:.4f}")
    print(f"Обработано кадров: {stats['total_frames']} (Face found in {stats['face_frames']})")
    print(f"Время обработки: {duration:.2f} сек ({(stats['total_frames']/duration):.1f} FPS)")
    
    return {"video": video_path, "score": score, "fps": stats['total_frames']/duration}

### запуск (change filename)

In [ ]:
video_to_test = "media/custom_filename.mp4"
result = process_video_prototype(video_to_test, config)